Les mathématiques de C4.5 : Le Ratio de GainPour calculer le Ratio de Gain, nous avons besoin d'une nouvelle métrique appelée Information de Division (Split Information). Elle mesure à quel point un attribut éparpille les données, indépendamment de la variable cible.Formule du **Ratio de Gain** :$$Gain\_Ratio(A) = \frac{Gain\_Information(A)}{Split\_Information(A)}$$

In [5]:
import math
from collections import Counter

# Calcul de l'entropie
def calculer_entropie(donnees, index_cible):
    valeurs_cibles = [ligne[index_cible] for ligne in donnees]
    compteur = Counter(valeurs_cibles)
    entropie = 0.0
    total = len(donnees)
    for nb in compteur.values():
        prob = nb / total
        entropie -= prob * math.log2(prob)
    return entropie

# Séparer les données 
def separer_donnees(donnees, index_attribut, valeur):
    return [ligne for ligne in donnees if ligne[index_attribut] == valeur]

# Calcul de l'Information de Division 
def information_division(donnees, index_attribut):
    valeurs = [ligne[index_attribut] for ligne in donnees]
    compteur = Counter(valeurs)
    total = len(donnees)
    
    split_info = 0.0
    for nb in compteur.values():
        prob = nb / total
        split_info -= prob * math.log2(prob)
        
    return split_info

#  Calcul du Ratio de Gain 
def ratio_gain(donnees, index_attribut, index_cible):
    # Calcul du gain d'information classique
    entropie_avant = calculer_entropie(donnees, index_cible)
    valeurs_uniques = set([ligne[index_attribut] for ligne in donnees])
    
    entropie_apres = 0.0
    total = len(donnees)
    
    for valeur in valeurs_uniques:
        sous_ensemble = separer_donnees(donnees, index_attribut, valeur)
        prob = len(sous_ensemble) / total
        entropie_apres += prob * calculer_entropie(sous_ensemble, index_cible)
        
    gain_info = entropie_avant - entropie_apres
    
    # Calcul du Split Information
    split_info = information_division(donnees, index_attribut)
    if split_info == 0:
        return 0.0
        
    return gain_info / split_info

In [6]:
def construire_arbre_c45(donnees, noms_attributs, index_cible):
    valeurs_cibles = [ligne[index_cible] for ligne in donnees]
    
    # Condition d'arrêt 1 : Données pures
    if len(set(valeurs_cibles)) == 1:
        return valeurs_cibles[0]
    
    # Condition d'arrêt 2 : Plus d'attributs
    if len(noms_attributs) == 0:
        return Counter(valeurs_cibles).most_common(1)[0][0]
    
    # Trouver le meilleur attribut en utilisant le RATIO DE GAIN 
    ratios = []
    for i in range(len(noms_attributs)):
        ratio = ratio_gain(donnees, i, index_cible)
        ratios.append(ratio)
        
    index_meilleur = ratios.index(max(ratios))
    meilleur_attribut = noms_attributs[index_meilleur]
    
    arbre = {meilleur_attribut: {}}
    valeurs_possibles = set([ligne[index_meilleur] for ligne in donnees])
    
    # Préparer les attributs restants
    attributs_restants = noms_attributs.copy()
    attributs_restants.pop(index_meilleur)
    
    for valeur in valeurs_possibles:
        sous_ensemble = separer_donnees(donnees, index_meilleur, valeur)
        
        # Enlever la colonne utilisée
        sous_ensemble_nettoye = []
        for ligne in sous_ensemble:
            nouvelle_ligne = ligne[:index_meilleur] + ligne[index_meilleur+1:]
            sous_ensemble_nettoye.append(nouvelle_ligne)
            
        # Gérer le cas où un sous-ensemble est vide
        if len(sous_ensemble_nettoye) == 0:
            arbre[meilleur_attribut][valeur] = Counter(valeurs_cibles).most_common(1)[0][0]
        else:
            nouvel_index_cible = index_cible - 1 if index_cible > index_meilleur else index_cible
            arbre[meilleur_attribut][valeur] = construire_arbre_c45(sous_ensemble_nettoye, attributs_restants, nouvel_index_cible)
        
    return arbre

In [4]:
import pprint

noms_colonnes = ["Jour_ID", "Meteo", "Temperature", "Vent", "Jouer_Tennis"]
donnees_entrainement = [
    ["J1", "Soleil", "Chaud", "Faible", "Non"],
    ["J2", "Soleil", "Chaud", "Fort", "Non"],
    ["J3", "Nuages", "Chaud", "Faible", "Oui"],
    ["J4", "Pluie", "Doux", "Faible", "Oui"],
    ["J5", "Pluie", "Frais", "Faible", "Oui"],
    ["J6", "Pluie", "Frais", "Fort", "Non"],
    ["J7", "Nuages", "Frais", "Fort", "Oui"],
    ["J8", "Soleil", "Doux", "Faible", "Non"],
    ["J9", "Soleil", "Frais", "Faible", "Oui"]
]

index_variable_cible = -1 

arbre_c45 = construire_arbre_c45(donnees_entrainement, noms_colonnes[:-1], index_variable_cible)

print("\nArbre de décision C4.5 généré :")
pprint.pprint(arbre_c45)

# Fonction de prédiction (Identique à ID3 car la structure de l'arbre final est la même)
def faire_prediction(arbre, ligne_donnee, noms_attributs):
    noeud_actuel = list(arbre.keys())[0]
    index_attribut = noms_attributs.index(noeud_actuel)
    valeur_donnee = ligne_donnee[index_attribut]
    
    try:
        sous_arbre = arbre[noeud_actuel][valeur_donnee]
    except KeyError:
        return "Valeur inconnue"
    
    if isinstance(sous_arbre, dict):
        return faire_prediction(sous_arbre, ligne_donnee, noms_attributs)
    else:
        return sous_arbre

# Test
# On définit les colonnes, mais on va dire à l'algorithme d'ignorer la première
noms_colonnes_complets = ["Jour_ID", "Meteo", "Temperature", "Vent", "Jouer_Tennis"]

# On ne passe à l'algorithme que les colonnes utiles (on saute l'index 0)
noms_attributs_utiles = ["Meteo", "Temperature", "Vent"]
donnees_sans_id = [ligne[1:] for ligne in donnees_entrainement]

# Construction de l'arbre C4.5 (sans les IDs inutiles)
arbre_c45_corrige = construire_arbre_c45(donnees_sans_id, noms_attributs_utiles, -1)

print("\nNouvel Arbre de décision (Généralisable) :")
pprint.pprint(arbre_c45_corrige)

# Test avec le nouveau jour (en ignorant aussi son ID J10)
nouveau_jour_sans_id = ["Soleil", "Frais", "Faible"]
prediction = faire_prediction(arbre_c45_corrige, nouveau_jour_sans_id, noms_attributs_utiles)

print(f"\nPrédiction pour {nouveau_jour_sans_id} : {prediction}")

Construction de l'arbre C4.5...

Arbre de décision C4.5 généré :
{'Jour_ID': {'J1': 'Non',
             'J2': 'Non',
             'J3': 'Oui',
             'J4': 'Oui',
             'J5': 'Oui',
             'J6': 'Non',
             'J7': 'Oui',
             'J8': 'Non',
             'J9': 'Oui'}}
Construction de l'arbre C4.5 (sans les IDs inutiles)...

Nouvel Arbre de décision (Généralisable) :
{'Meteo': {'Nuages': 'Oui',
           'Pluie': {'Vent': {'Faible': 'Oui', 'Fort': 'Non'}},
           'Soleil': {'Temperature': {'Chaud': 'Non',
                                      'Doux': 'Non',
                                      'Frais': 'Oui'}}}}

Prédiction pour ['Soleil', 'Frais', 'Faible'] : Oui
